# Final Report All-Gaps 24M Evidence Runner

Runs all final-report missing evidence groups from Colab after Google Drive is mounted and the repository is checked out. The normal top-to-bottom path performs a dry-run first, then runs all missing/unverified evidence without force-rerunning verified outputs.

Fixed protocol: train on `1990-01` to `1994-12`, then evaluate out-of-sample on `1995-01` to `1996-12` (`--test-months 24`) with `20` epochs, batch size `1024`, and seed `42`.

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone Or Update `main`

This cell clones `https://github.com/ROUCHER27/FYP.git` if `/content/FYP` does not already exist; otherwise it fast-forwards `main`.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/FYP
REPO_URL="https://github.com/ROUCHER27/FYP.git"

if [[ -d "$REPO_DIR/.git" ]]; then
  cd "$REPO_DIR"
  git fetch origin main
  git checkout main
  git pull --ff-only origin main
else
  if [[ -z "$REPO_URL" ]]; then
    echo "Set REPO_URL to your repository URL, then rerun this cell." >&2
    exit 1
  fi
  git clone --branch main "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
git status --short
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD

## Install Dependencies

In [ ]:
%%bash
set -euo pipefail
python -m pip install --upgrade pip
python -m pip install numpy pandas matplotlib seaborn scikit-learn torch

## Self-Check And Dry-Run

This prints every exact command the run-all cell will execute. It does not train.

In [ ]:
%%bash
set -euo pipefail
cd /content/FYP
bash -n scripts/run_final_report_all_24m_evidence_colab.sh
bash scripts/run_final_report_all_24m_evidence_colab.sh --dry-run

## Run All Evidence

This skips any run whose artifacts already verify. It writes only under `/content/drive/MyDrive/FYP/final_report_all_24m_evidence`.

In [ ]:
%%bash
set -euo pipefail
cd /content/FYP
bash scripts/run_final_report_all_24m_evidence_colab.sh

## Inspect Artifacts

In [ ]:
%%bash
set -euo pipefail
ROOT=/content/drive/MyDrive/FYP/final_report_all_24m_evidence
find "$ROOT" -maxdepth 3 -type f | sort | sed -n '1,200p'
echo
if [[ -f "$ROOT/reports/final_report_all_24m_evidence_status.txt" ]]; then
  cat "$ROOT/reports/final_report_all_24m_evidence_status.txt"
fi

## Optional Force Rerun

No executable force cell is included in the normal path. If you intentionally need to replace verified artifacts, run this manually from a new cell:

```bash
%%bash
set -euo pipefail
cd /content/FYP
bash scripts/run_final_report_all_24m_evidence_colab.sh --force
```